In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import datetime as dt
import sweetviz as sv
from sklearn.feature_extraction.text import TfidfVectorizer 

In [2]:
import zipfile
from pathlib import Path

# Nazwa pliku ZIP przesłanego do JupyterLab
zip_path = Path("real_estate_ads.csv.zip")

# Folder, do którego rozpakujemy archiwum
output_dir = Path("data")

output_dir.mkdir(parents=True, exist_ok=True)

# Rozpakowanie
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(output_dir)

print(f"Rozpakowano do: {output_dir.resolve()}")

# Wyświetlenie plików
for path in output_dir.rglob("*"):
    if path.is_file():
        print(f"{path}: {path.stat().st_size / 1024**2:.1f} MB")

Rozpakowano do: /home/sagemaker-user/data
data/real_estate_ads.csv: 36.2 MB
data/__MACOSX/._real_estate_ads.csv: 0.0 MB


In [3]:
np.random.seed(42)

plik = pd.read_csv("data/real_estate_ads.csv")
plik.head()

,id,market,created_at_first,created_at,district_lon,district_lat,title,description,params,price,no_rooms,m,price_per_m,map_lon,map_lat
0,49341,secondary,2018-01-04 13:11:05,2018-01-08 11:27:35,16.9408,52.38356,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>289000<br>price[curre...,289000.0,3,48.000000,6020.83,16.959516,52.391494
1,261720,secondary,2018-01-15 19:12:07,2018-01-25 11:11:31,16.9408,52.38356,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>379000<br>price[curre...,379000.0,3,61.000000,6213.11,16.950743,52.380535
2,50441,secondary,2018-01-04 13:53:29,2018-01-11 05:55:12,16.9408,52.38356,Rataje Nowe Miasto Niski Blok,<p>Weychan Nieruchomości Monika Hałas - Kurcza...,price<=>price<br>price<=>339000<br>price[curre...,339000.0,3,63.000000,5380.95,16.945860,52.393188
3,203518,secondary,2018-01-12 11:10:57,2018-01-30 14:38:39,16.9408,52.38356,"Mieszkanie 3 pokojowe blisko tramwaju, Poznań os.",<p>Oferuję na sprzedaż jasne i zadbane mieszka...,price<=>price<br>price<=>330000<br>price[curre...,330000.0,3,59.200001,5574.32,16.980279,52.391129
4,5,secondary,2018-01-02 11:34:51,2018-01-18 14:22:29,16.9408,52.38356,"Inwestycja -rondo Rataje student AWF,POLITECHNIKA",<p><strong>Dwa pokoje na niskim piętrze tuż pr...,price<=>price<br>price<=>259000<br>price[curre...,259000.0,2,38.000000,6815.79,16.950027,52.395861


In [4]:
df = pd.DataFrame(plik)
df["market"] = df["market"].map({
    "secondary": 0,
    "primary": 1
})

In [5]:
lat1 = np.radians(df["map_lat"])
lon1 = np.radians(df["map_lon"])

lat2 = np.radians(df["district_lat"])
lon2 = np.radians(df["district_lon"])

dlat = lat1 - lat2
dlon = lon1 - lon2

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)

df["dist_from_c"] = (
    6371 * 2 * np.arcsin(np.sqrt(a))
)

In [6]:
df = df.drop(columns=[
    "map_lon", "map_lat",
    "district_lon", "district_lat"
])

In [7]:
df = df[df["market"] == 0]
df.head()

,id,market,created_at_first,created_at,title,description,params,price,no_rooms,m,price_per_m,dist_from_c
0,49341,0,2018-01-04 13:11:05,2018-01-08 11:27:35,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>289000<br>price[curre...,289000.0,3,48.000000,6020.83,1.546445
1,261720,0,2018-01-15 19:12:07,2018-01-25 11:11:31,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>379000<br>price[curre...,379000.0,3,61.000000,6213.11,0.754062
2,50441,0,2018-01-04 13:53:29,2018-01-11 05:55:12,Rataje Nowe Miasto Niski Blok,<p>Weychan Nieruchomości Monika Hałas - Kurcza...,price<=>price<br>price<=>339000<br>price[curre...,339000.0,3,63.000000,5380.95,1.124327
3,203518,0,2018-01-12 11:10:57,2018-01-30 14:38:39,"Mieszkanie 3 pokojowe blisko tramwaju, Poznań os.",<p>Oferuję na sprzedaż jasne i zadbane mieszka...,price<=>price<br>price<=>330000<br>price[curre...,330000.0,3,59.200001,5574.32,2.808298
4,5,0,2018-01-02 11:34:51,2018-01-18 14:22:29,"Inwestycja -rondo Rataje student AWF,POLITECHNIKA",<p><strong>Dwa pokoje na niskim piętrze tuż pr...,price<=>price<br>price<=>259000<br>price[curre...,259000.0,2,38.000000,6815.79,1.504283


In [8]:
report = sv.analyze(df)
report.show_html('report.html', open_browser = False)

                                             |          | [  0%]   00:00 -> (? left)

Report report.html was generated.


In [9]:
df.created_at = df.created_at.apply(lambda x: dt.datetime.strptime(x, "%Y-%m-%d %H:%M:%S"))

In [10]:
df.created_at_first = df.created_at_first.apply(lambda x: dt.datetime.strptime(x, "%Y-%m-%d %H:%M:%S"))

In [11]:
df.sort_values(['created_at'])


,id,market,created_at_first,created_at,title,description,params,price,no_rooms,m,price_per_m,dist_from_c
8,87213,0,2018-01-05 17:10:49,2018-01-06 22:01:29,"Poznań, os. Piastowskie, 2 pok. niski blok",<p><br/>Doskonała inwestycja w bardzo dobrej l...,price<=>price<br>price<=>199000<br>price[curre...,199000.0,2,38.000000,5236.84,3.370614
0,49341,0,2018-01-04 13:11:05,2018-01-08 11:27:35,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>289000<br>price[curre...,289000.0,3,48.000000,6020.83,1.546445
2,50441,0,2018-01-04 13:53:29,2018-01-11 05:55:12,Rataje Nowe Miasto Niski Blok,<p>Weychan Nieruchomości Monika Hałas - Kurcza...,price<=>price<br>price<=>339000<br>price[curre...,339000.0,3,63.000000,5380.95,1.124327
6,58549,0,2018-01-04 16:02:43,2018-01-11 05:55:12,Oświecenia 1 Piętro !!! Niski Blok 289 tys.,<p>Weychan Nieruchomości Monika Hałas-Kurczaba...,price<=>price<br>price<=>289000<br>price[curre...,289000.0,3,48.000000,6020.83,1.515437
68,100157,0,2018-01-07 03:44:11,2018-01-12 05:31:29,"Mieszkanie 119 m2, Apartamentowiec, Poznań- Ra...",<p><br/><strong>SPRZEDAŻ. MIESZKANIE. APARTAME...,price<=>price<br>price<=>2380000<br>price[curr...,2380000.0,3,119.000000,20000.00,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
13520,9592283,0,2021-01-24 22:10:19,2021-01-31 10:20:26,Okazja 1 piętro w niskim bloku,<p><br/>Zadbane mieszkanie po kapitalnym remon...,price<=>price<br>price<=>419900<br>price[curre...,419900.0,3,48.000000,8747.92,2.460401
13546,9575200,0,2021-01-20 02:00:26,2021-01-31 10:20:26,Okazja 1 piętro w niskim bloku,<p><br/>Zadbane mieszkanie po kapitalnym remon...,price<=>price<br>price<=>419900<br>price[curre...,419900.0,3,48.000000,8747.92,0.687886
13568,9506878,0,2021-01-01 12:10:49,2021-01-31 12:13:03,Rataje wyremontowane 2 pokoje + meble i sprzęty,<p>Sprzedam bezpośrednio ładne mieszkanie na 4...,price<=>price<br>price<=>299999<br>price[curre...,299999.0,2,38.000000,7894.71,0.876521
10521,8905455,0,2020-07-20 14:48:12,2021-01-31 18:22:14,Ustawne mieszkanie os. Rzeczypospolitej Poznań,<p>Bardzo ustawne mieszkanie 3 pokojowe z osob...,price<=>price<br>price<=>380000<br>price[curre...,380000.0,3,47.200001,8050.85,0.715528


In [12]:
df_numerical = df[[k for k,v in df.dtypes.to_dict().items() if v == 'float64']]

In [13]:
df_numerical

,price,m,price_per_m,dist_from_c
0,289000.0,48.000000,6020.83,1.546445
1,379000.0,61.000000,6213.11,0.754062
2,339000.0,63.000000,5380.95,1.124327
3,330000.0,59.200001,5574.32,2.808298
4,259000.0,38.000000,6815.79,1.504283
...,...,...,...,...
13729,290000.0,38.000000,7631.58,0.701612
13733,323000.0,32.240002,10018.61,1.975298
13738,425000.0,58.000000,7327.59,2.318068
13739,280000.0,38.000000,7368.42,0.000000


In [14]:
df.dtypes.to_dict()

{'id': dtype('int64'),
 'market': dtype('int64'),
 'created_at_first': dtype('<M8[ns]'),
 'created_at': dtype('<M8[ns]'),
 'title': dtype('O'),
 'description': dtype('O'),
 'params': dtype('O'),
 'price': dtype('float64'),
 'no_rooms': dtype('O'),
 'm': dtype('float64'),
 'price_per_m': dtype('float64'),
 'dist_from_c': dtype('float64')}

In [15]:
np.unique(df.no_rooms.values)

array(['1', '2', '3', '4', '5', '6', '7', 'more'], dtype=object)

In [16]:
def temporal_split(df, split_ratio = 0.8):
    df_test = df.iloc[int(split_ratio*len(df)):,:]
    df_train = df.iloc[:int(split_ratio*len(df)),:]
    return df_train, df_test

In [17]:
df_numerical =df_numerical.dropna()
df_numerical_test = df_numerical.iloc[int(0.8*len(df_numerical)):,:]
df_numerical_train = df_numerical.iloc[:int(0.8*len(df_numerical)),:]

In [18]:
df_numerical_test

,price,m,price_per_m,dist_from_c
9991,595000.0,67.300003,8841.01,2.581029
10006,278000.0,26.000000,10692.31,1.930511
10020,399000.0,61.200001,6519.61,0.000000
10021,306000.0,38.000000,8052.63,0.801442
10022,549000.0,80.599998,6811.41,3.310907
...,...,...,...,...
13729,290000.0,38.000000,7631.58,0.701612
13733,323000.0,32.240002,10018.61,1.975298
13738,425000.0,58.000000,7327.59,2.318068
13739,280000.0,38.000000,7368.42,0.000000


In [19]:
# BASELINE LINEAR REGRESSION

In [20]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

x_train = df_numerical_train[["m", "price_per_m", "dist_from_c"]]
y_train = df_numerical_train[['price']]
x_test = df_numerical_test[["m", "price_per_m", "dist_from_c"]]
y_test = df_numerical_test[["price"]]


lr_pipe = Pipeline([('scaler',StandardScaler()),('lr', LinearRegression())])

In [21]:
lr_pipe.fit(x_train,y_train)

,steps,"[('scaler', ...), ('lr', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


In [22]:
y_hat = lr_pipe.predict(x_test)

In [23]:
from sklearn.metrics import mean_absolute_error as mae, mean_squared_error as mse, root_mean_squared_error as rmse

In [24]:
errors = {('LinearRegression', 'mae'): mae(y_test, y_hat), ('LinearRegression', 'mse'): mse(y_test, y_hat), 
          ('LinearRegression', 'rmse'): rmse(y_test, y_hat)}


In [25]:
errors

{('LinearRegression', 'mae'): 80764.72202951633,
 ('LinearRegression', 'mse'): 15098553475.098677,
 ('LinearRegression', 'rmse'): 122876.17130712804}

In [26]:
df_numerical_with_rooms = df[["m", "price", "no_rooms", "price_per_m","dist_from_c"]]

df_numerical_with_rooms.no_rooms = df_numerical_with_rooms.no_rooms.apply(lambda x:int(x) if x!='more' else 8.5)

df_numerical_with_rooms["dist_from_c"] = df["dist_from_c"].fillna(df_numerical_with_rooms["dist_from_c"].mean())

df_numerical_with_rooms_train, df_numerical_with_rooms_test = temporal_split(df_numerical_with_rooms, 0.8)


/tmp/ipykernel_9363/3172629910.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_numerical_with_rooms.no_rooms = df_numerical_with_rooms.no_rooms.apply(lambda x:int(x) if x!='more' else 8.5)
/tmp/ipykernel_9363/3172629910.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_numerical_with_rooms["dist_from_c"] = df["dist_from_c"].fillna(df_numerical_with_rooms["dist_from_c"].mean())


In [27]:
x_train = df_numerical_with_rooms_train[["m", "price_per_m", "dist_from_c","no_rooms"]]
y_train = df_numerical_with_rooms_train[['price']]
x_test = df_numerical_with_rooms_test[["m", "price_per_m", "dist_from_c", "no_rooms"]]
y_test = df_numerical_with_rooms_test[["price"]]
lr_pipe = Pipeline([('scaler',StandardScaler()),('lr', LinearRegression())])


In [28]:
lr_pipe.fit(x_train,y_train)

,steps,"[('scaler', ...), ('lr', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


In [29]:
y_hat = lr_pipe.predict(x_test)

In [30]:
errors = {('LinearRegression_ROOMS', 'mae'): mae(y_test, y_hat), ('LinearRegression_ROOMS', 'mse'): mse(y_test, y_hat), 
          ('LinearRegression_ROOMS', 'rmse'): rmse(y_test, y_hat)}

In [31]:
errors

{('LinearRegression_ROOMS', 'mae'): 58147.89341640335,
 ('LinearRegression_ROOMS', 'mse'): 8542570499.2202015,
 ('LinearRegression_ROOMS', 'rmse'): 92426.02717427706}

In [ ]:
df_full = df.dropna()
df_full

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(title)